In [1]:
# 02-torgo-parameterization

# Author: Eryk Urbański
# Date: May 2025
# Description: TORGO Dataset Parameterization

# Using the plap library for parameterization

In [32]:
import time
import os
import sys
sys.path.append("..")

import plap
from dataset_loader import TORGO

import pandas as pd
import soundfile as sf

### Defining Feature Vector

In [ ]:
fvector_all = plap.FeatureVector(
    "asc","asc_var",
    "ass","ass_var",
    "asf","asf_mean","asf_var","asf_var_mean",
    "aff","aff_var",
    "lat","tc",
    "sc","sc_var",
    "hsc","hsd","hss","hsv",
    "asb","asb_mean","asp","asp_mean",
    "mfcc"
    )

In [ ]:
# Feature names compatibility taking into account vector features
feature_names_compat = [feature.upper() for feature in fvector_all.features]

# Define mapping for base features and their corresponding counts
feature_mapping = {
    'ASF': 24,
    'ASF_VAR': 24,
    'ASB': 20,
    'ASP': 21,
    'MFCC': 20
}

i = 0
while i < len(feature_names_compat):
    base_feature = feature_names_compat[i]
    
    if base_feature in feature_mapping:
        count = feature_mapping[base_feature]
        feature_names_compat[i] = f'{base_feature}1'  # Set the first feature name
        
        # Create the numbered features list
        numbered_features = [f'{base_feature}{k}' for k in range(2, count + 1)]
        
        # Insert numbered features right after the base feature
        feature_names_compat[i+1:i+1] = numbered_features
        
        # Skip over the newly inserted features
        i += len(numbered_features)
    
    i += 1

feature_names_compat

In [ ]:
len(feature_names_compat)

### Defining Preprocessor instances

In [ ]:
preprocessor1 = plap.Preprocessor(preemphasis_coeff=None, block_size=256, window_type="hamming")
preprocessor2_preemph = plap.Preprocessor(preemphasis_coeff=0.68, block_size=256, window_type="hamming")

### Parameterization

In [ ]:
torgo_path = "E:\\STUDIA\\semestr 8\\WSI\\Projekt\\Data\\TORGO" # Adjust as needed
torgo = TORGO(path=torgo_path)

In [ ]:
def batch_process_torgo(torgo: TORGO, batch_size, fvector: plap.FeatureVector, preprocessor: plap.Preprocessor, log_csv_path="parameterization_log.csv"):
    columns = ['path', 'label'] + feature_names_compat
    successful_df = pd.DataFrame(columns=columns)
    failed_to_parameterize = []

    # Create or overwrite log file initially
    with open(log_csv_path, 'w') as f:
        f.write("batch_idx,batch_time_seconds,num_success,num_failed\n")

    total = len(torgo)
    iterator = 0
    batch_idx = 0
    
    while iterator < total:

        # if batch_idx == 2: # optional stopping condition, adjust/remove if needed
        #     break

        start_time = time.time()

        batch_audio_paths, batch_labels = torgo.load_batch(iterator, batch_size)

        batch_records = []
        batch_failed = 0

        for path, label in zip(batch_audio_paths, batch_labels):
            try:
                plap.parameterize(path, fvector, preprocessor)
                features = fvector.values
                row = [path, label] + features.tolist()
                batch_records.append(row)
            except Exception as e:
                failed_to_parameterize.append(path)
                batch_failed += 1

        if batch_records:
            columns = ['path', 'label'] + feature_names_compat
            df = pd.DataFrame(batch_records, columns=columns)
            # df = pd.DataFrame(batch_records)
            successful_df = pd.concat([successful_df, df], ignore_index=True)

        batch_time = time.time() - start_time
        num_success = len(batch_records)

        # Append log entry to CSV
        with open(log_csv_path, 'a') as f:
            f.write(f"{batch_idx},{batch_time:.2f},{num_success},{batch_failed}\n")

        batch_idx += 1
        iterator += batch_size

    return successful_df

In [ ]:
files_in_batch = 50
n_batches = len(torgo) // files_in_batch + 1
n_batches

In [ ]:
df = batch_process_torgo(torgo, files_in_batch, fvector_all, preprocessor1)

In [ ]:
df.shape

In [ ]:
pd.set_option('display.max_columns', None)  # Show all columns
df.head()

In [ ]:
df[df.isna().any(axis=1)]

In [ ]:
df.to_csv('torgo_nopreemph_hamming256.csv')

# Add audio lengths to csv

In [8]:
def add_audio_lengths_to_csv_in_chunks(csv_path, output_path, chunk_size=500):
    first_chunk = True
    reader = pd.read_csv(csv_path, chunksize=chunk_size)

    for chunk_idx, chunk in enumerate(reader):
        new_rows = []

        for _, row in chunk.iterrows():
            path = row['path']
            try:
                audio, sr = sf.read(path)
                length_samp = len(audio)
                length_sec = len(audio) / sr
            except Exception as e:
                print(f"[Chunk {chunk_idx}] Failed to read {path}: {e}")
                length_samp = None
                length_sec = None

            # Reconstruct row with new columns inserted after 'path'
            row_dict = row.to_dict()
            row_items = list(row_dict.items())
            path_idx = next(i for i, (k, _) in enumerate(row_items) if k == 'path')

            new_row_items = (
                row_items[:path_idx + 1] +
                [('length_samples', length_samp), ('length_seconds', length_sec)] +
                row_items[path_idx + 1:]
            )
            new_rows.append(dict(new_row_items))

        # Convert to DataFrame and save
        result_df = pd.DataFrame(new_rows)

        if first_chunk:
            result_df.to_csv(output_path, index=False, mode='w')  # overwrite if exists
            first_chunk = False
        else:
            result_df.to_csv(output_path, index=False, mode='a', header=False)  # append

    print(f"\n✅ Finished processing. Output saved to: {output_path}")

In [10]:
add_audio_lengths_to_csv_in_chunks(csv_path="torgo_nopreemph_hamming256.csv", output_path="torgo_nopreemph_hamming256_lengths.csv", chunk_size=500)


✅ Finished processing. Output saved to: torgo_nopreemph_hamming256_lengths.csv


# Add original prompts from TORGO to csv

In [56]:
def add_prompt_column(csv_path, output_path, chunk_size=500):
    """
    For each row, adds a 'prompt' column by extracting the filename from the 'path' column
    and reading the corresponding prompt file.
    Saves the processed output in chunks to avoid high memory usage.
    """
    # Create or overwrite the output file with headers
    headers_written = False
    for chunk in pd.read_csv(csv_path, chunksize=chunk_size):
        prompts = []

        for idx, row in chunk.iterrows():
            try:
                # Extract the filename (e.g., 'wav_arrayMic_FC01S01_0001.wav')
                filename = os.path.basename(row['path'])
                # Get just the number part (e.g., '0001')
                prompt_id = filename.split('_')[-1].split('.')[0]

                # Get speaker/session info from the filename
                parts = filename.split('_')
                if len(parts[2]) > 5:
                    speaker_id = parts[2][:4] if 'C' in parts[2] else parts[2][:3] # e.g., 'FC01' or 'M01'
                    session_number = parts[2][-1] # e.g., 'S01' → '1'
                else:
                    speaker_id = parts[2]
                    session_number = '1'
                speaker_g = (speaker_id[0] + 'C') if 'C' in speaker_id else speaker_id[0]

                session_number = '2_3' if speaker_id=='M01' else session_number # exception

                # Construct the prompt path
                prompt_path = os.path.join(
                    r"E:/STUDIA/semestr 8/WSI/Projekt/Data",
                    speaker_g,
                    speaker_id,
                    f"Session{session_number}",
                    "prompts",
                    f"{prompt_id}.txt"
                )

                # Read the content of the prompt file
                with open(prompt_path, 'r', encoding='utf-8') as f:
                    prompt_text = f.read().strip()
                prompts.append(prompt_text)

            except Exception as e:
                print(f"Failed to load prompt for row {idx} (file {filename}): {e}")
                prompts.append(None)

        # Add the prompt column to the chunk
        chunk.insert(loc=chunk.columns.get_loc('path') + 1, column='prompt', value=prompts)

        # Append to output file
        if not headers_written:
            chunk.to_csv(output_path, index=False, mode='w')
            headers_written = True
        else:
            chunk.to_csv(output_path, index=False, mode='a', header=False)

    print(f"Finished processing prompts. Output saved to: {output_path}")


In [ ]:
add_prompt_column(csv_path="torgo_nopreemph_hamming256_lengths.csv", output_path="torgo_nopreemph_hamming256_lengths_prompts.csv", chunk_size=500)

# Add prompt categories

In [61]:
def categorize_prompt(prompt: str) -> str:
    if pd.isna(prompt):
        return 'undefined'

    prompt = prompt.strip().lower()

    # Rule 6: exception first
    if 'as in' in prompt:
        return 'short_word'

    if any(x in prompt for x in ['"', '.']) or len(prompt.split()) > 2:
        return 'restricted_sentence'
    elif '.jpg' in prompt:
        return 'unrestricted_sentence'
    elif len(prompt.split()) == 1:
        if len(prompt) == 1 or '[say' in prompt or '[relax' in prompt:
            return 'non_words'
        else:
            return 'short_word'
    else:
        return 'undefined'


def add_prompt_category_column(csv_path, output_path, chunk_size=500):
    """
    Adds a 'prompt_category' column to the CSV based on the 'prompt' content.
    Inserts the column directly after 'prompt'.
    Saves the result in chunks to avoid memory issues.
    """
    headers_written = False

    for chunk in pd.read_csv(csv_path, chunksize=chunk_size):
        prompt_cat = chunk['prompt'].apply(categorize_prompt)
        insert_index = chunk.columns.get_loc('prompt') + 1
        chunk.insert(insert_index, 'prompt_category', prompt_cat)

        if not headers_written:
            chunk.to_csv(output_path, index=False, mode='w')
            headers_written = True
        else:
            chunk.to_csv(output_path, index=False, mode='a', header=False)

    print(f"Prompt categories saved to: {output_path}")

In [62]:
add_prompt_category_column(csv_path="torgo_nopreemph_hamming256_lengths_prompts.csv", output_path="torgo_nopreemph_hamming256_lengths_prompts_categories.csv", chunk_size=500)

Prompt categories saved to: torgo_nopreemph_hamming256_lengths_prompts_categories.csv
